# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

### Lane Selected: **Refresh / Content Opportunity Scoring** (Core Lane 2)

**Why this lane?**

1. **The starter dataset is built for it.** The repo ships `data/raw/content_refresh_anonymized.csv` (~30,000 rows), and the entire starter pipeline (`scripts/01`–`05`) is already a working end-to-end example of this exact problem. This means I can iterate from a verified baseline instead of starting from zero.

2. **The decision is real and constrained.** Content teams have limited review capacity. They cannot refresh every page. The question "which pages should we review first?" is a prioritization problem with a clear action (review → refresh/expand/protect/prune/monitor) and a measurable cost of getting it wrong.

3. **The baseline is beatable, but not trivially.** The starter results show baseline rules achieve Precision@50 = 0.240 (about 12 correct out of top 50), while a random forest reaches 0.740 (about 37 correct). That is a ~3× improvement in reviewer productivity — but it is earned on a 30K-row slice, not the full 79M-row warehouse. There is room to build a stronger label, a better feature window, and more honest validation on the warehouse data.

4. **I am not interested in "train a model" as an end goal.** I am interested in whether a ranked queue with reason codes can help a human reviewer spend their limited time on pages where evidence of decline or opportunity is strongest. The model is a tool; the decision support is the deliverable.



# Week 1: Research Question & Problem Framing

## 1) My Lane and Why
I am choosing **Lane 1: Content Prioritization (Ranking/Scoring)** to identify which content items are at risk of traffic decline and need immediate editorial review.

Instead of treating this as a simple classification problem ("will it decline?"), treating it as a ranking problem directly serves the business decision of where to allocate limited editor hours. By identifying the top $K$ items most vulnerable to performance drops, we optimize resource deployment.

Our exploration of the starter dataset below provides the baseline numbers that justify why this lane is both viable and valuable for the next 7 weeks.

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

> **Which content pages should be reviewed first for refresh, expansion, protection, pruning, or monitoring — based on observable search and engagement signals from the prior 90 days?**

### Unit of Analysis
One row = **one content page** (deduplicated by `content_id`). The grain is page-level, not daily. Features are aggregated over a 90-day window; the target is a future-window outcome (e.g., decline over the next 30 days) or a proxy label derived from the current window.

### The Decision This Improves
A content strategist or SEO reviewer decides **which pages to spend their limited time on today.** Without a ranked queue, they either:
- Review pages in random order (inefficient), or
- Use simple rules like "stale and visible" (better, but misses nuance and interaction effects).

A scored queue lets them start with the pages where the evidence of opportunity or risk is strongest.

### The Action Someone Takes
From the ranked output, a reviewer inspects the top-K pages and chooses one of:
- **Refresh** — update outdated facts, stats, or examples;
- **Expand** — add depth where the page is thin relative to demand;
- **Protect** — monitor a high-performing page showing early decay signals;
- **Prune** — consider consolidation if the page is redundant or cannibalizing another;
- **Monitor** — no action now, but flag for re-check in the next cycle.

Each page carries **reason codes** (e.g., `declining_with_demand`, `stale_visible_page`, `low_ctr_visible_page`) so the reviewer understands *why* it surfaced.

### Cost of a Wrong Recommendation

| Type | What happens | Cost |
|------|-------------|------|
| **False Positive** | We flag a page for review, but it does not need changes. | Reviewer time is wasted. Opportunity cost: another page that *did* need help was pushed down the queue. |
| **False Negative** | We miss a page that is genuinely declining. | Traffic and revenue loss continues until the decline is noticed manually. For high-impression pages, this can be significant. |
| **Wrong Action** | We recommend "refresh" but the page actually needs "prune/merge." | Effort is spent editing a page that should have been consolidated. May create duplicate content issues. |

The baseline wastes ~38 out of every 50 review slots (P@50 = 0.240). Improving that to even 0.500 means 25 more correct pages reviewed per batch. At scale, that is a massive efficiency gain.

### Why Data / ML Can Help Here
- **Signal is noisy and high-dimensional.** A page's "need" depends on impressions, clicks, CTR, position, age, freshness, engagement, scroll rate, word count, content type, intent, and trend direction — plus their interactions. A human cannot hold all of this in working memory for 500,000 pages.
- **Patterns exist but are not obvious.** The random forest beating the baseline by 3× on the starter slice suggests that interaction effects (e.g., "old + high impressions + declining CTR") carry signal that simple rules miss.
- **The decision is repetitive.** The same prioritization happens every week. A model that is validated, monitored, and updated is more consistent than ad-hoc human judgment at scale.
- **The output is decision-support, not automation.** The model ranks; the human decides. This keeps the human in the loop and avoids overclaiming.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [33]:
import pandas as pd
import numpy as np

# Load the starter dataset
df = pd.read_csv('content_refresh_anonymized.csv')

# 1. Total records and unique clients
total_rows = len(df)
unique_clients = df['client_id'].nunique()


In [7]:
print(total_rows,unique_clients)

30000 32


In [9]:
df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [10]:
df.columns


Index(['content_id', 'client_id', 'search_volume', 'competition',
       'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count',
       'char_count', 'provider_used', 'model_used', 'impressions_90d',
       'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d',
       'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
       'days_with_impressions', 'days_with_sessions', 'impressions_last_30d',
       'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d',
       'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier',
       'age_tier_order', 'days_since_last_update', 'freshness_tier',
       'word_count_tier', 'char_count_tier', 'ctr', 'avg_position',
       'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier',
       'position_tier', 'trend_direction', 'trend_pct'],
      dtype='object')

In [34]:
declined_df=df[df['trend_direction']=='down']

declining_count = declined_df['trend_direction'].count()

decline_rate = (declining_count / total_rows) * 100

# 3. Check missingness or anomalies (e.g., avg_position == 0 means "no data")
no_rank_data = (df['avg_position'] == 0).sum()

print(f"--- Starter Dataset Insights ---")
print(f"Total Content Items (Rows): {total_rows:,}")
print(f"Unique Clients: {unique_clients}")
print(f"Declining Items Count: {declining_count:,} ({decline_rate:.2f}%)")
print(f"Items with No Rank Data (avg_position == 0): {no_rank_data:,}")

--- Starter Dataset Insights ---
Total Content Items (Rows): 30,000
Unique Clients: 32
Declining Items Count: 16,262 (54.21%)
Items with No Rank Data (avg_position == 0): 1,205


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*


### What I CAN Claim
- **"This model ranks pages by the probability of exhibiting decline signals, based on observable search and engagement metrics from a prior 90-day window."**
- **"On the starter validation set, the top 50 ranked pages contained ~37 true positives versus ~12 for the baseline rule."**
- **"The ranked queue is a decision-support tool: it suggests which pages to review first, not which pages are guaranteed to recover after editing."**
- **"Results are observational. We measured associations between past signals and current labels, not causal effects of refreshes on traffic."**
- **"Validation used client-holdout: the model was tested on clients it never saw during training, which tests generalizability to new accounts."**

### What I CANNOT Claim (and will never write)
- ❌ "This model predicts which pages will recover after a refresh." (We have no causal design; we do not know what would have happened without the refresh.)
- ❌ "These are the factors Google uses to rank pages." (We observe correlations, not algorithm internals.)
- ❌ "The model is 74% accurate." (Precision@50 is not accuracy. It is the fraction of the top 50 that are true positives. The overall label prevalence and false negative rate matter too.)
- ❌ "AI platforms cannot understand this content." (We only have AI session counts, not AI citation or ranking data.)
- ❌ "Every page with a high score needs a refresh." (The score means "review this first," not "edit this blindly." Human judgment is still required.)

### What I Will Do to Stay Honest
1. **Define a future-window label** (e.g., decline in next 30 days) instead of relying on the current-window `trend_direction == "down"` proxy. The proxy is convenient but not ideal because it uses the same window for features and label.
2. **Audit for leakage** before every model run: check that no feature uses information from the target window, and that no derived field encodes the target.
3. **Report by position tier** when discussing CTR, because CTR is not comparable across positions 1 and 10 without adjustment.
4. **Show the top 20 by hand** in the final notebook: read the actual rows, inspect the reason codes, and flag any that look like noise or seasonality.
5. **Compare against the baseline** every time. A model that does not beat a transparent rule is not worth deploying.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

## 5) Self-Check

Before I move to Week 2, I answer the checklist from the assignment card:

| Check | Answer |
|-------|--------|
| **Did I pick one of the four predefined lanes or declare freestyle?** | ✅ Core Lane 2: Refresh / Content Opportunity Scoring. |
| **Did I name the decision?** | ✅ "Which pages should the content team review first today?" |
| **Did I name the action?** | ✅ Review → refresh, expand, protect, prune, or monitor. Each page gets reason codes. |
| **Did I explain the cost of a wrong call?** | ✅ False positive = wasted reviewer time; false negative = continued traffic loss; wrong action = editing instead of pruning. |
| **Did I show at least two real numbers from the starter data?** | TotalContent Items (Rows): 30,000; Unique Clients: 32; Declining Items Count: 16,262 (54.21%); Items with No Rank Data (avg_position == 0): 1,205
 |
| **Did I explain why this is not just "train a model"?** | ✅ The model is a prioritization tool. The deliverable is a ranked queue with reason codes that helps humans make better decisions with limited time. The baseline is interpretable; the model must beat it honestly. |
| **Did I use careful language?** | ✅ Distinguish ranking from causation, precision@50 from accuracy, and decision-support from automation. |

### Open Questions for Week 2
1. **Label strength:** Can I build a future-window label (e.g., 90-day features → 30-day decline) on the warehouse data, or do I start with the proxy and upgrade later?
2. **Feature engineering:** Which interactions matter most? (e.g., `age * impressions`, `position * CTR`)
3. **Validation design:** Client-holdout worked on the starter slice. On the warehouse, should I use time-aware split (train on older months, test on newer) or keep client-holdout?
4. **Threshold policy:** The starter uses 80th percentile + minimum volume filters. Should I tune this by reviewing the top 20 manually and measuring reviewer capacity?

---

**Next step:** Build the data contract (Week 2) and run the signal audit on the warehouse sample.